In [ ]:
# | hide


In [ ]:
# | hide
#| eval: false
from dotenv import load_dotenv
import os

load_dotenv()
# BASE_PATH = os.environ["SEO_RAT_BASE_PATH"]
BASE_PATH = os.environ["AWAZLY_PATH"]
# BASE_PATH = os.environ["emdadelgaz_PATH"]
# BASE_PATH = os.environ["EMDAD_PATH"]
# BASE_PATH = "/home/kobo/Desktop/astro_hacking/ainclean"

In [ ]:

#| eval: false
#| hide
from pathlib import Path


def get_url_file_mapping_ain() -> dict[str, str]:
    urls = fetch_sitemap_urls("https://www.alainclean.com/sitemap.xml")
    mapping = map_all_urls_to_files(
        base_path="/home/kobo/Desktop/astro_hacking/ainclean/articles",
        site_url="https://www.alainclean.com/article",
        urls=[u for u in urls if "/article/" in u],
        mode="direct"
    )

    # mark unmapped URLs for fetching
    for url in urls:
        if url not in mapping:
            mapping[url] = FETCH
    return mapping


def get_url_file_mapping_smaa() -> dict[str, str]:
    from urllib.parse import urlparse, parse_qs, unquote
    base = "/home/kobo/Desktop/astro_hacking/samma_garden/articles"
    urls = fetch_sitemap_urls("https://www.smaagarden.com/sitemap.xml")
    mapping = {}
    for url in urls:
        qs = parse_qs(urlparse(url).query)
        if slug := qs.get("slug", [None])[0]:
            path = f"{base}/{unquote(slug)}.md"
            mapping[url] = path if Path(path).exists() else FETCH
        else:
            mapping[url] = FETCH
    return mapping


def get_url_file_mapping_gpuvec() -> dict[str, str]:
    from seo_rat.content_mapper import map_all_urls_to_files
    from seo_rat.index_tracking import fetch_sitemap_urls
    from seo_rat.report.articles import FETCH

    urls = fetch_sitemap_urls("https://gpuvec.com/sitemap.xml")

    mapping = map_all_urls_to_files(
        base_path="/home/kobo/Desktop/Desktop/gpuvec",
        site_url="https://gpuvec.com",
        urls=urls,
        mode="direct"
    )

    for url in urls:
        if url not in mapping:
            mapping[url] = FETCH
    return mapping




In [ ]:

#| eval: false
#| hide
with get_session() as session:
    websites = session.exec(select(Website)).all()
    for w in websites:
        print(f"ID: {w.id}, URL: {w.url}")

    urls = fetch_sitemap_urls("https://kareemai.com/sitemap.xml")
    # url_mapping = map_all_urls_to_files(
    #     base_path=f"{BASE_PATH}",
    #     site_url="https://www.smaagarden.com",
    #     urls=urls,
    #     mode="direct",
    # )
    url_mapping = map_all_urls_to_files(
        base_path=f"{BASE_PATH}",
        site_url="https://kareemai.com/",
        urls=urls,
        mode="direct"
    )
    sync_articles_to_db(session, website_id=4, url_file_mapping=url_mapping)
    articles = session.exec(select(Article).where(Article.website_id == 4)).all()
    for article in articles:
        print(article)


# SEO Report
> Sync articles, detect duplicate metadata, and generate SEO reports from GSC data.

## Delete Articles in wrong Website

In [ ]:
#| hide
#| eval: false
# from sqlmodel import delete
# session.exec(
#     delete(Article).where(
#         Article.website_id == 2,
#         Article.file_path.contains("emdadgaz")
#     )
# )
# session.commit()


In [ ]:
#| hide
#| eval: false
from seo_rat.models import Website
from seo_rat.models import GSCAnalytics
from sqlalchemy import func

session.exec(select(Website)).all()
session.exec(
    select(GSCAnalytics.site_url, func.count().label("rows"))
    .group_by(GSCAnalytics.site_url)
).all()


In [ ]:
# | hide
#| eval: false
website = Website(url="https://alainclean.com/", name="alainclean", lang="ar",
                  desc="موقع متخصص في خدمات تنظيف املنازل في العين والإمارات")

with get_session() as session:
    session.add(website)
    session.commit()
    session.refresh(website)
    print(website.id)


In [ ]:
#| hide
#| eval: false
from seo_rat.models import get_all_websites, print_websites

with get_session() as session:
    websites = get_all_websites(session)
    print_websites(websites)

## Make FocusKeyword to be Top query from GSC


In [ ]:
from seo_rat.gsc.sync import sync_missing_dates
# auth.authenticate()
#| hide
#| eval: false
from seo_rat.gsc_client import GSCAuth
from seo_rat.gsc.sync import daily_sync

start, end = get_date_range("last_days", days=31 * 4)
print(start, end)
auth = GSCAuth(secrets_file="client_secrets.json")
with get_session() as session:
    sync_missing_dates(session, auth, site_url="sc-domain:kareemai.com", start_date=start, end_date=end)


In [ ]:
#| hide
#| eval: false
website_name = "https://awazly.com/"

with get_session() as session:
    articles = get_articles_by_website(session, website_id=1)
    print(len(articles))
    for article in articles:
        page_path = article.url.removeprefix(website_name)
        print(page_path)
        print(article)
        top_query = get_top_queries(session=session, site_url="sc-domain:kareemai.com", start_date=start,
                                    end_date=end, page_path=page_path, limit=1, sort_by="impressions")
        top_5_query = get_top_queries(session=session, site_url="sc-domain:kareemai.com", start_date=start,
                                      end_date=end, page_path=page_path, limit=5, sort_by="impressions")
        try:
            print(top_query[0]['query'])
            article.focus_keyword = top_query[0]['query']
            article.secondary_keywords = [q["query"] for q in top_5_query[:5]]
            session.add(article)
            session.commit()
        except IndexError:
            print("No queries found for this page")


In [ ]:
# | hide
#| eval: false

from seo_rat.content_parser import remove_metadata

with get_session() as session:
    article = session.exec(select(Article)).first()
    print(article)
    with open(article.file_path, "r") as f:
        content = remove_metadata(f.read())

    link_analysis = analyze_links(content, "gpuvec.com")
    print(link_analysis)

    print(f"Total: {link_analysis['total_links']}")
    print(f"Internal: {link_analysis['internal_count']}")
    print(f"External: {link_analysis['external_count']}")


In [ ]:
#| hide
#|eval: false
article = Article(website_id=4, file_path="::fetch::", url="https://kareemai.com")
analyze_article(article, "kareemai.com", is_quarto=True)


In [ ]:
#| hide
#| eval: false
from seo_rat.content_parser import get_markdown_files
from seo_rat.article import get_articles_by_website
from seo_rat.content.analysis import find_cannibalized


In [ ]:
#| hide
#| eval: false
articles = get_articles_by_website(session, website_id=1)
print(len(articles))


In [ ]:
#| hide
#| eval: false
start, end = get_date_range("last_days", days=200)
# find_cannibalized(session, 8, "sc-domain:gpuvec.com", start, end)



In [ ]:
# | hide
# | eval: false
report = generate_seo_report(
    session,
    website_id=1,
    domain="awazly.com",
    is_quarto=False,
    title_is_h1=True,
    desc_field="description",
    date_field="date",
    fetch_base_url="http://localhost:4321/",
)
print(f"Total pages: {report['total_pages']}")
print(f"Issues found: {report['summary']['total_issues']}")


In [ ]:
# | hide
#| eval: false
report


In [ ]:
#| hide
#| eval: false
print_issues_report(report)


In [ ]:
#| hide
#| eval: false
report['issues']['https://emdadelgaz.com/ahwadh-sbahh-bnzam-alawfr-flw']